# GBDF Gender Labels — Download + Commit

Small standalone notebook: grabs the single `GBDF_training_labels.xlsx` (gender annotations,
the accuracy-gap fairness baseline) from the GBDF GitHub release, verifies it, and commits.

GBDF is gender-only — your **primary** fairness labels are the Xu 65.3M annotations (already downloaded:
A-FF++, A-Celeb-DF with age/gender/ethnicity). GBDF is just the baseline-for-contrast your plan calls for.


## Cell 1 — Mount + paths + git creds

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, subprocess
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
PARENT = "/content/drive/MyDrive/CDTS_Research"
ANNO = f"{REPO}/data/annotations/GBDF"
os.makedirs(ANNO, exist_ok=True)
# restore git creds from Drive
for f in [".git-credentials", ".gitconfig"]:
    src = f"{PARENT}/{f}"
    if os.path.exists(src): subprocess.run(f'cp "{src}" /root/{f}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
print("REPO exists:", os.path.isdir(REPO))
print("GBDF folder:", ANNO)

Mounted at /content/drive
REPO exists: True
GBDF folder: /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/data/annotations/GBDF


## Cell 2 — Download the GBDF xlsx

Tries the standard release-asset filename. If it fails, the asset is named slightly differently —
check https://github.com/aakash4305/GBDF/releases/tag/v1.0 under "Assets" for the exact name.


In [2]:
import os, subprocess, glob
# candidate filenames (README says GBDF_training_labels.xlsx; try variants)
candidates = [
    "GBDF_training_labels.xlsx",
    "GBDF_testing_labels.xlsx",
    "GBDF_labels.xlsx",
]
base = "https://github.com/aakash4305/GBDF/releases/download/v1.0"
got = []
for fname in candidates:
    out = f"{ANNO}/{fname}"
    subprocess.run(f'wget -q -O "{out}" "{base}/{fname}"', shell=True)
    sz = os.path.getsize(out) if os.path.exists(out) else 0
    if sz > 1000:
        print(f"OK  {fname}: {sz/1024:.0f} KB")
        got.append(fname)
    else:
        print(f"--  {fname}: not found ({sz} bytes)")
        if os.path.exists(out): os.remove(out)
print("\nGBDF xlsx files:", [os.path.basename(f) for f in glob.glob(f"{ANNO}/*.xlsx")])
if not got:
    print("\n>>> None downloaded. Open the release page, check Assets for the exact xlsx name,")
    print(">>> then edit the candidates list above with the correct filename.")

OK  GBDF_training_labels.xlsx: 174 KB
--  GBDF_testing_labels.xlsx: not found (0 bytes)
--  GBDF_labels.xlsx: not found (0 bytes)

GBDF xlsx files: ['GBDF_training_labels.xlsx']


## Cell 3 — Inspect the xlsx (confirm it's real gender-label data)

In [3]:
import glob, os
import pandas as pd
xlsx = glob.glob(f"{ANNO}/*.xlsx")
if xlsx:
    df = pd.read_excel(xlsx[0])
    print("file:", os.path.basename(xlsx[0]))
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("\nfirst rows:")
    print(df.head(5).to_string())
else:
    print("no xlsx to inspect - run Cell 2 first")

file: GBDF_training_labels.xlsx
shape: (6999, 5)
columns: ['Filename', 'Unnamed: 1', 'Dataset', 'Real', 'Gender']

first rows:
       Filename  Unnamed: 1   Dataset  Real Gender
0  id0_0000.mp4         NaN  Celeb-DF     1      M
1  id0_0002.mp4         NaN  Celeb-DF     1      M
2  id0_0003.mp4         NaN  Celeb-DF     1      M
3  id0_0004.mp4         NaN  Celeb-DF     1      M
4  id0_0005.mp4         NaN  Celeb-DF     1      M


## Cell 4 — Gitignore + commit

The xlsx is small (likely <1MB) so it CAN go in git. But annotations are already gitignored
(`data/annotations/`), so we force-add this one file with `-f`, OR just leave GBDF Drive-only
like the Xu annotations. Default here: commit it (it's tiny and useful to version).


In [4]:
import os, glob, subprocess
os.chdir(REPO)
xlsx = glob.glob(f"{ANNO}/*.xlsx")
if xlsx:
    # force-add past the data/annotations/ gitignore (file is tiny)
    for x in xlsx:
        subprocess.run(f'git add -f "{x}"', shell=True)
    subprocess.run('git status --short', shell=True)
    print("\n(review staged above; run Cell 5 to commit)")
else:
    print("no xlsx - nothing to commit")


(review staged above; run Cell 5 to commit)


## Cell 5 — Commit + push

In [5]:
import os, subprocess
os.chdir(REPO)
subprocess.run('git commit -m "Add GBDF gender labels (accuracy-gap fairness baseline)"', shell=True)
subprocess.run('git push origin main 2>&1 | tail -4', shell=True)
print("\ncommit hash:")
subprocess.run('git rev-parse HEAD', shell=True)


commit hash:


CompletedProcess(args='git rev-parse HEAD', returncode=0)

In [6]:
import os, subprocess
os.chdir(REPO)
r = subprocess.run("git log --oneline -2", shell=True, capture_output=True, text=True)
print("=== recent commits ===")
print(r.stdout)
r2 = subprocess.run("git status --short", shell=True, capture_output=True, text=True)
print("=== uncommitted ===")
print(r2.stdout if r2.stdout.strip() else "(clean - all committed)")
r3 = subprocess.run("git log --oneline origin/main -1", shell=True, capture_output=True, text=True)
print("=== latest on GitHub ===")
print(r3.stdout)

=== recent commits ===
ae1d6bf Add Celeb-DF-v2 test split: 518 videos (16420 frames, 124 identities) + leakage-safe manifest + JSON
8eb1803 Close NB01: complete FF++ c23 dataset (159627 frames, 1000 identities) + leakage-safe manifest; fix build_manifest_from_json method extraction; capture DeepfakeBench patches

=== uncommitted ===
 m external/DeepfakeBench
 M notebooks/Celeb-DF_crop.ipynb
?? data/df40_core/
?? notebooks/GBDF_download.ipynb

=== latest on GitHub ===
ae1d6bf Add Celeb-DF-v2 test split: 518 videos (16420 frames, 124 identities) + leakage-safe manifest + JSON



In [7]:
import os, subprocess, glob
os.chdir(REPO)

# 1) gitignore df40_core (50GB zips - Drive only)
with open(".gitignore", "a") as f:
    f.write("\n# DF40 core method copy - large zips, Drive-only\ndata/df40_core/\n")

# 2) find the GBDF xlsx
xlsx = glob.glob(f"{REPO}/data/annotations/GBDF/*.xlsx")
print("GBDF xlsx found:", [os.path.basename(x) for x in xlsx])

# 3) force-add the xlsx (past data/annotations/ ignore) + add notebooks + gitignore
for x in xlsx:
    r = subprocess.run(f'git add -f "{x}"', shell=True, capture_output=True, text=True)
    if r.stderr: print("add error:", r.stderr)
subprocess.run("git add .gitignore notebooks/GBDF_download.ipynb notebooks/Celeb-DF_crop.ipynb", shell=True)

# 4) show what's staged
r = subprocess.run("git status --short", shell=True, capture_output=True, text=True)
print("\n=== staged/status ===")
print(r.stdout)

GBDF xlsx found: ['GBDF_training_labels.xlsx']

=== staged/status ===
M  .gitignore
 m external/DeepfakeBench
M  notebooks/Celeb-DF_crop.ipynb
A  notebooks/GBDF_download.ipynb



In [8]:
import os, subprocess
os.chdir(REPO)
xlsx_rel = "data/annotations/GBDF/GBDF_training_labels.xlsx"
xlsx_abs = f"{REPO}/{xlsx_rel}"

# does git see it as ignored, and why?
r = subprocess.run(f'git check-ignore -v "{xlsx_rel}"', shell=True, capture_output=True, text=True)
print("check-ignore (shows which rule blocks it):", r.stdout if r.stdout else "not ignored")

# try force-add with the RELATIVE path (sometimes absolute paths confuse git)
r2 = subprocess.run(f'git add -f "{xlsx_rel}"', shell=True, capture_output=True, text=True)
print("force-add stdout:", r2.stdout if r2.stdout else "(none)")
print("force-add stderr:", r2.stderr if r2.stderr else "(none)")

# is it staged now?
r3 = subprocess.run(f'git status --short "{xlsx_rel}"', shell=True, capture_output=True, text=True)
print("status for xlsx:", r3.stdout if r3.stdout else "(not showing - still not staged)")

# confirm file really exists where we think
print("file exists:", os.path.exists(xlsx_abs), f"({os.path.getsize(xlsx_abs)} bytes)" if os.path.exists(xlsx_abs) else "")

check-ignore (shows which rule blocks it): .gitignore:22:data/annotations/	data/annotations/GBDF/GBDF_training_labels.xlsx

force-add stdout: (none)
force-add stderr: (none)
status for xlsx: (not showing - still not staged)
file exists: True (178522 bytes)


In [9]:
import os, subprocess
os.chdir(REPO)
# df40_core gitignore + notebooks already staged from before; just commit them
r = subprocess.run('git commit -m "Add GBDF + Celeb-DF download notebooks; gitignore df40_core (GBDF xlsx stays Drive-only like Xu annotations)"',
                   shell=True, capture_output=True, text=True)
print(r.stdout); print(r.stderr if r.stderr else "")
r2 = subprocess.run("git push origin main", shell=True, capture_output=True, text=True)
print(r2.stdout[-400:] if r2.stdout else ""); print(r2.stderr[-400:] if r2.stderr else "")
subprocess.run("git log --oneline -1", shell=True)
r3 = subprocess.run("git log --oneline -1", shell=True, capture_output=True, text=True)
print("now at:", r3.stdout)

[main 6d50d65] Add GBDF + Celeb-DF download notebooks; gitignore df40_core (GBDF xlsx stays Drive-only like Xu annotations)
 3 files changed, 5 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/GBDF_download.ipynb



To https://github.com/anasbiswas1/deepfake-trust-research.git
   ae1d6bf..6d50d65  main -> main

now at: 6d50d65 Add GBDF + Celeb-DF download notebooks; gitignore df40_core (GBDF xlsx stays Drive-only like Xu annotations)



In [10]:
import os, glob
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB = f"{REPO}/external/DeepfakeBench"

# 1) where are the checkpoints, and which detectors do we have configs for?
print("=== checkpoints ===")
for f in glob.glob(f"{REPO}/weights/**/*.pth", recursive=True)[:20]:
    print(f"  {os.path.getsize(f)/1e6:.0f}MB  {f.replace(REPO+'/','')}")

print("\n=== DeepfakeBench detector configs ===")
for f in sorted(glob.glob(f"{DFB}/training/config/detector/*.yaml")):
    print("  ", os.path.basename(f))

print("\n=== test harness ===")
print("test.py exists:", os.path.exists(f"{DFB}/training/test.py"))
for f in glob.glob(f"{DFB}/training/*.py"):
    print("  ", os.path.basename(f))

Mounted at /content/drive
=== checkpoints ===

=== DeepfakeBench detector configs ===
   altfreezing.yaml
   capsule_net.yaml
   clip.yaml
   core.yaml
   efficientnetb4.yaml
   effort.yaml
   f3net.yaml
   facexray.yaml
   ffd.yaml
   ftcn.yaml
   fwa.yaml
   i3d.yaml
   iid.yaml
   lrl.yaml
   lsda.yaml
   meso4.yaml
   meso4Inception.yaml
   multi_attention.yaml
   pcl_xception.yaml
   recce.yaml
   resnet34.yaml
   rfm.yaml
   sbi.yaml
   sia.yaml
   sladd_detector.yaml
   spsl.yaml
   srm.yaml
   stil.yaml
   tall.yaml
   timetransformer.yaml
   ucf.yaml
   uia_vit.yaml
   videomae.yaml
   xception.yaml
   xclip.yaml

=== test harness ===
test.py exists: True
   logger.py
   test.py
   train.py


In [11]:
import os, glob
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# search broadly for any model weights anywhere in the project
print("=== all .pth/.pt files in project ===")
hits = glob.glob(f"{REPO}/**/*.pth", recursive=True) + glob.glob(f"{REPO}/**/*.pt", recursive=True)
for f in hits[:40]:
    print(f"  {os.path.getsize(f)/1e6:>6.0f}MB  {f.replace(REPO+'/','')}")
print(f"\ntotal weight files: {len(hits)}")

# also check the df40_checkpoints folder specifically
print("\n=== weights/ tree ===")
for root, dirs, files in os.walk(f"{REPO}/weights"):
    depth = root[len(f"{REPO}/weights"):].count(os.sep)
    if depth <= 2:
        print(f"  {root.replace(REPO+'/','')}/  ({len(files)} files, {len(dirs)} subdirs)")

=== all .pth/.pt files in project ===

total weight files: 0

=== weights/ tree ===
  weights/  (0 files, 0 subdirs)


In [12]:
import os, glob
# search ALL of Drive, not just the repo, for the df40 checkpoints
print("=== searching all of Drive for checkpoints ===")
roots_to_check = [
    "/content/drive/MyDrive/CDTS_Research",
    "/content/drive/MyDrive",
]
for root in roots_to_check:
    hits = glob.glob(f"{root}/**/*.pth", recursive=True)
    print(f"\n{root}: {len(hits)} .pth files")
    for h in hits[:15]:
        print(f"  {os.path.getsize(h)/1e6:.0f}MB  {h.replace('/content/drive/MyDrive/','')}")

# also check for a df40_checkpoints or df40_weights folder anywhere
print("\n=== checkpoint folders ===")
for pat in ["*df40_checkpoint*", "*df40_weight*", "*checkpoint*", "*weights*"]:
    for d in glob.glob(f"/content/drive/MyDrive/**/{pat}", recursive=True):
        if os.path.isdir(d):
            n = len(glob.glob(f"{d}/**/*.pth", recursive=True))
            print(f"  {d.replace('/content/drive/MyDrive/','')}: {n} .pth")

=== searching all of Drive for checkpoints ===

/content/drive/MyDrive/CDTS_Research: 0 .pth files

/content/drive/MyDrive: 0 .pth files

=== checkpoint folders ===
  XIDS_Research/xids-research/models/seed_runs/seed123/_checkpoints: 0 .pth
  XIDS_Research/xids-research/models/seed_runs/seed456/_checkpoints: 0 .pth
  XIDS_Research/xids-research/models/seed_runs/seed789/_checkpoints: 0 .pth
  CDTS_Research/deepfake-trust-research/weights: 0 .pth


In [13]:
import os, glob
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# find the checkpoint shortcut
hits = set(glob.glob("/content/drive/MyDrive/**/df40_*weight*", recursive=True) +
           glob.glob("/content/drive/MyDrive/**/df40_*checkpoint*", recursive=True) +
           glob.glob("/content/drive/MyDrive/CDTS_Research/*", recursive=False))
print("=== candidate checkpoint folders ===")
for h in sorted(hits):
    if os.path.isdir(h):
        pths = glob.glob(f"{h}/**/*.pth", recursive=True)
        if pths:
            print(f"\n{h.replace('/content/drive/MyDrive/','')}: {len(pths)} checkpoints")
            for p in pths[:15]:
                print(f"  {os.path.getsize(p)/1e6:>5.0f}MB  {p.split('/')[-2]}/{os.path.basename(p)}")

Mounted at /content/drive
=== candidate checkpoint folders ===

CDTS_Research/df40_weights: 61 checkpoints
     88MB  train_on_fs_matrix/blendface_ff.pth
     88MB  train_on_fs_matrix/e4s_ff.pth
     88MB  train_on_fs_matrix/facedancer_ff.pth
     88MB  train_on_fs_matrix/faceswap_ff.pth
     88MB  train_on_fs_matrix/fsgan_ff.pth
     88MB  train_on_fs_matrix/inswap_ff.pth
     88MB  train_on_fs_matrix/mobileswap_ff.pth
     88MB  train_on_fs_matrix/simswap_ff.pth
     88MB  train_on_fs_matrix/uniface_ff.pth
    343MB  train_on_fr/clip.pth
   1213MB  train_on_fr/clip_large.pth
    109MB  train_on_fr/i3d.pth
    191MB  train_on_fr/recce.pth
     88MB  train_on_fr/rfm.pth
     88MB  train_on_fr/spsl.pth


In [14]:
import os, glob
SC = "/content/drive/MyDrive/CDTS_Research/df40_weights"
print("=== checkpoint tree by training-set ===")
for tier in sorted(os.listdir(SC)):
    tdir = f"{SC}/{tier}"
    if os.path.isdir(tdir):
        pths = glob.glob(f"{tdir}/*.pth")
        print(f"\n{tier}/ ({len(pths)} checkpoints):")
        for p in sorted(pths):
            print(f"  {os.path.getsize(p)/1e6:>5.0f}MB  {os.path.basename(p)}")

=== checkpoint tree by training-set ===

train_on_df40/ (4 checkpoints):
    343MB  clip.pth
   1213MB  clip_large.pth
    109MB  i3d.pth
     88MB  xception.pth

train_on_efs/ (7 checkpoints):
    343MB  clip.pth
   1213MB  clip_large.pth
    191MB  recce.pth
     88MB  rfm.pth
     88MB  spsl.pth
    222MB  srm.pth
     88MB  xception.pth

train_on_efs_matrix/ (11 checkpoints):
     88MB  DiT_ff.pth
     88MB  SiT_ff.pth
     88MB  StyleGAN2_ff.pth
     88MB  StyleGAN3_ff.pth
     88MB  StyleGANXL_ff.pth
     88MB  VQGAN_ff.pth
     88MB  ddim_ff.pth
     88MB  pixart_ff.pth
     88MB  rddm_ff.pth
     88MB  sd2.1_ff.pth
     88MB  wav2lip_ff.pth

train_on_fr/ (8 checkpoints):
    343MB  clip.pth
   1213MB  clip_large.pth
    109MB  i3d.pth
    191MB  recce.pth
     88MB  rfm.pth
     88MB  spsl.pth
    222MB  srm.pth
     88MB  xception.pth

train_on_fr_matrix/ (12 checkpoints):
     88MB  MRAA_ff.pth
     88MB  danet_ff.pth
     88MB  facevid2vid_ff.pth
     88MB  fomm_ff.pth
     

In [15]:
import os, shutil, glob
SC = "/content/drive/MyDrive/CDTS_Research/df40_weights/train_on_fs"
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DST = f"{REPO}/weights/train_on_fs"
os.makedirs(DST, exist_ok=True)

# the NB02 roster: Xception (Tier1), CLIP (Tier4), EfficientNetB4 (Tier1), F3Net (Tier2)
roster = ["xception.pth", "clip.pth", "efficientnetb4.pth", "f3net.pth"]
for ckpt in roster:
    src, dst = f"{SC}/{ckpt}", f"{DST}/{ckpt}"
    if os.path.exists(src) and not os.path.exists(dst):
        print(f"copying {ckpt} ({os.path.getsize(src)/1e6:.0f}MB)...")
        shutil.copy2(src, dst)
    elif os.path.exists(dst):
        print(f"{ckpt} already copied")
    else:
        print(f"{ckpt} NOT in source")

print("\n=== weights/train_on_fs now ===")
for p in sorted(glob.glob(f"{DST}/*.pth")):
    print(f"  {os.path.getsize(p)/1e6:>5.0f}MB  {os.path.basename(p)}")

copying xception.pth (88MB)...
copying clip.pth (343MB)...
copying efficientnetb4.pth (71MB)...
copying f3net.pth (90MB)...

=== weights/train_on_fs now ===
    343MB  clip.pth
     71MB  efficientnetb4.pth
     90MB  f3net.pth
     88MB  xception.pth


In [16]:
import os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB = f"{REPO}/external/DeepfakeBench"

# the xception detector config — tells us model class, input size, preprocessing
print("=== xception.yaml ===")
print(open(f"{DFB}/training/config/detector/xception.yaml").read())

=== xception.yaml ===
# log dir 
log_dir: /data/home/zhiyuanyan/DeepfakeBench/logs/testing_bench

# model setting
pretrained: ./training/pretrained/xception-b5690688.pth   # path to a pre-trained model, if using one
model_name: xception   # model name
backbone_name: xception  # backbone name

#backbone setting
backbone_config:
  mode: original
  num_classes: 2
  inc: 3
  dropout: false

# dataset
all_dataset: [FaceForensics++, FF-F2F, FF-DF, FF-FS, FF-NT, FaceShifter, DeepFakeDetection, Celeb-DF-v1, Celeb-DF-v2, DFDCP, DFDC, DeeperForensics-1.0, UADFV]
train_dataset: [FaceForensics++]
test_dataset: [FaceForensics++, DeepFakeDetection]

compression: c23  # compression-level for videos
train_batchSize: 32   # training batch size
test_batchSize: 32   # test batch size
workers: 8   # number of data loading workers
frame_num: {'train': 32, 'test': 32}   # number of frames to use per video in training and testing
resolution: 256   # resolution of output image to network
with_mask: false   # 

In [ ]:
import os, glob, torch
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB = f"{REPO}/external/DeepfakeBench"

# 1) detector implementations available
print("=== detectors/ ===")
for f in sorted(glob.glob(f"{DFB}/training/detectors/*.py")):
    print("  ", os.path.basename(f))

# 2) the xception detector class — how it's built
xd = f"{DFB}/training/detectors/xception_detector.py"
if os.path.exists(xd):
    print(f"\n=== xception_detector.py (first 60 lines) ===")
    print("".join(open(xd).readlines()[:60]))

# 3) CRITICAL: what's the actual key structure in your checkpoint?
ckpt = f"{REPO}/weights/train_on_fs/xception.pth"
sd = torch.load(ckpt, map_location="cpu")
print(f"\n=== checkpoint structure ===")
print("top-level type:", type(sd))
if isinstance(sd, dict):
    keys = list(sd.keys())
    print("top-level keys (first 5):", keys[:5])
    # is it wrapped (e.g. {'state_dict': ...}) or raw weights?
    if 'state_dict' in sd:
        print("-> wrapped in 'state_dict'")
        keys = list(sd['state_dict'].keys())
    print("first 5 weight keys:", keys[:5])
    print("last 3 weight keys:", keys[-3:])

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_2984/2099140337.py", line 1, in <cell line: 0>
    import os, glob, torch
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtra

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
print("mount alive:", os.path.isdir(REPO))
print("checkpoints still there:", os.path.exists(f"{REPO}/weights/train_on_fs/xception.pth"))

Mounted at /content/drive
mount alive: True
checkpoints still there: True


In [2]:
import os, glob, torch
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB = f"{REPO}/external/DeepfakeBench"

# 1) detector implementations available
print("=== detectors/ ===")
for f in sorted(glob.glob(f"{DFB}/training/detectors/*.py")):
    print("  ", os.path.basename(f))

# 2) the xception detector class — how it's built
xd = f"{DFB}/training/detectors/xception_detector.py"
if os.path.exists(xd):
    print(f"\n=== xception_detector.py (first 60 lines) ===")
    print("".join(open(xd).readlines()[:60]))

# 3) CRITICAL: actual key structure in your checkpoint
ckpt = f"{REPO}/weights/train_on_fs/xception.pth"
sd = torch.load(ckpt, map_location="cpu")
print(f"\n=== checkpoint structure ===")
print("top-level type:", type(sd))
if isinstance(sd, dict):
    keys = list(sd.keys())
    print("top-level keys (first 5):", keys[:5])
    if 'state_dict' in sd:
        print("-> wrapped in 'state_dict'")
        keys = list(sd['state_dict'].keys())
    print("first 5 weight keys:", keys[:5])
    print("last 3 weight keys:", keys[-3:])

=== detectors/ ===
   __init__.py
   altfreezing_detector.py
   base_detector.py
   capsule_net_detector.py
   clip_detector.py
   core_detector.py
   efficientnetb4_detector.py
   effort_detector.py
   f3net_detector.py
   facexray_detector.py
   ffd_detector.py
   ftcn_detector.py
   fwa_detector.py
   i3d_detector.py
   iid_detector.py
   lrl_detector.py
   lsda_detector.py
   meso4Inception_detector.py
   meso4_detector.py
   multi_attention_detector.py
   pcl_xception_detector.py
   recce_detector.py
   resnet34_detector.py
   rfm_detector.py
   sbi_detector.py
   sia_detector.py
   sladd_detector.py
   spsl_detector.py
   srm_detector.py
   sta_detector.py
   stil_detector.py
   tall_detector.py
   timesformer_detector.py
   ucf_detector.py
   uia_vit_detector.py
   videomae_detector.py
   xception_detector.py
   xclip_detector.py

=== xception_detector.py (first 60 lines) ===
'''
# author: Zhiyuan Yan
# email: zhiyuanyan@link.cuhk.edu.cn
# date: 2023-0706
# description: Class fo

In [3]:
import os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
xd = f"{REPO}/external/DeepfakeBench/training/detectors/xception_detector.py"
src = open(xd).read()
# show the forward method and what it returns
import re
fwd = src[src.find("def forward"):]
print("=== forward method ===")
print(fwd[:1200])

=== forward method ===
def forward(self, data_dict: dict, inference=False) -> dict:
        # get the features by backbone
        features = self.features(data_dict)
        # get the prediction by classifier
        pred = self.classifier(features)
        # get the probability of the pred
        prob = torch.softmax(pred, dim=1)[:, 1]
        # build the prediction dict for each output
        pred_dict = {'cls': pred, 'prob': prob, 'feat': features}
        return pred_dict



In [4]:
import os
DFB = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/external/DeepfakeBench"
print(open(f"{DFB}/training/detectors/__init__.py").read())

import os
import sys
current_file_path = os.path.abspath(__file__)
parent_dir = os.path.dirname(os.path.dirname(current_file_path))
project_root_dir = os.path.dirname(parent_dir)
sys.path.append(parent_dir)
sys.path.append(project_root_dir)

from metrics.registry import DETECTOR
from .utils import slowfast

from .facexray_detector import FaceXrayDetector
from .xception_detector import XceptionDetector
from .efficientnetb4_detector import EfficientDetector
from .resnet34_detector import ResnetDetector
from .f3net_detector import F3netDetector
from .meso4_detector import Meso4Detector
from .meso4Inception_detector import Meso4InceptionDetector
from .spsl_detector import SpslDetector
from .core_detector import CoreDetector
from .capsule_net_detector import CapsuleNetDetector
from .srm_detector import SRMDetector
from .ucf_detector import UCFDetector
from .recce_detector import RecceDetector
from .fwa_detector import FWADetector
from .ffd_detector import FFDDetector
from .videomae_detector

In [5]:
import sys, os, importlib.util
DFB = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/external/DeepfakeBench"
TRAIN = f"{DFB}/training"
for p in [TRAIN, DFB]:
    if p not in sys.path: sys.path.insert(0, p)

# install the LIGHT deps xception_detector itself needs (not the video stack)
!pip -q install efficientnet_pytorch timm einops 2>&1 | tail -1

# register the metrics.registry DETECTOR (xception_detector imports `from detectors import DETECTOR`)
# we pre-create the 'detectors' module with just DETECTOR so the import resolves without running __init__
import types
from metrics.registry import DETECTOR
fake_pkg = types.ModuleType("detectors")
fake_pkg.DETECTOR = DETECTOR
fake_pkg.__path__ = [f"{TRAIN}/detectors"]   # so submodule imports still resolve
sys.modules["detectors"] = fake_pkg

# now import ONLY the xception detector module file directly
spec = importlib.util.spec_from_file_location(
    "detectors.xception_detector", f"{TRAIN}/detectors/xception_detector.py")
xmod = importlib.util.module_from_spec(spec)
sys.modules["detectors.xception_detector"] = xmod
spec.loader.exec_module(xmod)
XceptionDetector = xmod.XceptionDetector
print("XceptionDetector loaded, bypassing package __init__:", XceptionDetector)

/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/external/DeepfakeBench/training/detectors/xception_detector.py:21: SyntaxWarning: invalid escape sequence '\s'
  author={Rossler, Andreas and Cozzolino, Davide and Verdoliva, Luisa and Riess, Christian and Thies, Justus and Nie{\ss}ner, Matthias},


ModuleNotFoundError: No module named 'kornia'

In [6]:
!pip -q install kornia 2>&1 | tail -1

# retry loading xception
import importlib.util, sys
TRAIN = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/external/DeepfakeBench/training"
try:
    spec = importlib.util.spec_from_file_location(
        "detectors.xception_detector", f"{TRAIN}/detectors/xception_detector.py")
    xmod = importlib.util.module_from_spec(spec)
    sys.modules["detectors.xception_detector"] = xmod
    spec.loader.exec_module(xmod)
    XceptionDetector = xmod.XceptionDetector
    print("OK - XceptionDetector loaded:", XceptionDetector)
except ModuleNotFoundError as e:
    print("next missing:", e.name)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 88.3 MB/s eta 0:00:00
OK - XceptionDetector loaded: <class 'detectors.xception_detector.XceptionDetector'>


In [3]:
import importlib.util, sys, os
DFB = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/external/DeepfakeBench"
for p in [f"{DFB}/training", DFB]:
    if p not in sys.path: sys.path.insert(0, p)

# install the few LIGHT deps the xception path actually needs (not fvcore/slowfast)
!pip -q install efficientnet_pytorch timm 2>&1 | tail -1

# import ONLY the xception detector module directly, not the whole detectors package
from detectors.xception_detector import XceptionDetector
print("XceptionDetector imported directly:", XceptionDetector)

ModuleNotFoundError: No module named 'fvcore'

In [2]:
import os, sys, glob
import numpy as np, torch, torch.nn.functional as F
from PIL import Image
import pandas as pd
from sklearn.metrics import roc_auc_score

REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB = f"{REPO}/external/DeepfakeBench"
CKPT = f"{REPO}/weights/train_on_fs/xception.pth"

# make DeepfakeBench importable
for p in [f"{DFB}/training", DFB]:
    if p not in sys.path: sys.path.insert(0, p)

config = {"model_name":"xception","backbone_name":"xception",
    "backbone_config":{"mode":"original","num_classes":2,"inc":3,"dropout":False},
    "pretrained":None,"resolution":256,"loss_func":"cross_entropy",
    "mean":[0.5,0.5,0.5],"std":[0.5,0.5,0.5]}

from detectors import DETECTOR
model = DETECTOR["xception"](config)

sd = torch.load(CKPT, map_location="cpu")
new_sd = {(k[7:] if k.startswith("module.") else k): v for k,v in sd.items()}
missing, unexpected = model.load_state_dict(new_sd, strict=False)
print(f"loaded | missing:{len(missing)} unexpected:{len(unexpected)}")
if len(missing)>5: print("  WARN missing:", missing[:5])
if len(unexpected)>5: print("  WARN unexpected:", unexpected[:5])

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print("device:", device)

MEAN, STD, RES = np.array(config["mean"],np.float32), np.array(config["std"],np.float32), 256
def load_frame(path):
    img = Image.open(path).convert("RGB").resize((RES,RES), Image.BILINEAR)
    arr = (np.asarray(img,np.float32)/255.0 - MEAN)/STD
    return torch.from_numpy(np.transpose(arr,(2,0,1)))

mani = pd.read_parquet(f"{REPO}/data/ffpp_manifest.parquet")
test = mani[mani["split"]=="test"] if "split" in mani.columns else mani
reals = test[test.label==0].sample(n=min(100,(test.label==0).sum()), random_state=42)
fakes = test[test.label==1].sample(n=min(100,(test.label==1).sum()), random_state=42)
sample = pd.concat([reals,fakes]).reset_index(drop=True)
print(f"sampled {len(sample)} ({(sample.label==0).sum()} real, {(sample.label==1).sum()} fake)")

probs, labels = [], []
with torch.no_grad():
    for i in range(0, len(sample), 32):
        rows = sample.iloc[i:i+32]
        imgs, labs = [], []
        for _, r in rows.iterrows():
            if os.path.exists(r["frame_path"]):
                imgs.append(load_frame(r["frame_path"])); labs.append(int(r["label"]))
        if not imgs: continue
        out = model({"image": torch.stack(imgs).to(device)}, inference=True)
        probs.extend(out["prob"].cpu().numpy().tolist()); labels.extend(labs)

probs, labels = np.array(probs), np.array(labels)
auc = roc_auc_score(labels, probs)
print(f"\n{'='*40}")
print(f"  IN-DOMAIN FF++ AUC: {auc:.4f}  (n={len(labels)})")
print(f"  fake-prob: [{probs.min():.3f},{probs.max():.3f}] mean {probs.mean():.3f}")
print(f"{'='*40}")
print("  PASS - foundation solid" if auc>=0.85 else
      "  MARGINAL - check preprocessing" if auc>=0.65 else
      "  FAIL - debug load/preprocessing")

ModuleNotFoundError: No module named 'fvcore'

In [9]:
import os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
CKPT = f"{REPO}/weights/train_on_fs/xception.pth"
print("exists:", os.path.exists(CKPT))
print("size:", os.path.getsize(CKPT)/1e6, "MB" if os.path.exists(CKPT) else "")
print("readable:", os.access(CKPT, os.R_OK))
# try opening raw
try:
    with open(CKPT, "rb") as f:
        head = f.read(16)
    print("first bytes:", head[:8])
except Exception as e:
    print("open failed:", e)

exists: True
size: 87.767115 MB
readable: True
first bytes: b'PK\x03\x04\x00\x00\x08\x08'


In [11]:
import os, io, glob
import numpy as np, torch
from PIL import Image
import pandas as pd
from sklearn.metrics import roc_auc_score

REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
CKPT = f"{REPO}/weights/train_on_fs/xception.pth"

config = {"model_name":"xception","backbone_name":"xception",
    "backbone_config":{"mode":"original","num_classes":2,"inc":3,"dropout":False},
    "pretrained":None,"resolution":256,"loss_func":"cross_entropy",
    "mean":[0.5,0.5,0.5],"std":[0.5,0.5,0.5]}

# build model
model = XceptionDetector(config)

# load checkpoint via buffer (avoids Drive mount seek issue)
with open(CKPT, "rb") as f:
    buf = io.BytesIO(f.read())
sd = torch.load(buf, map_location="cpu")
new_sd = {(k[7:] if k.startswith("module.") else k): v for k,v in sd.items()}
missing, unexpected = model.load_state_dict(new_sd, strict=False)
print(f"loaded | missing:{len(missing)} unexpected:{len(unexpected)}")
if len(missing)>5: print("  WARN missing:", missing[:8])
if len(unexpected)>5: print("  WARN unexpected:", unexpected[:8])

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print("device:", device)

MEAN, STD, RES = np.array([0.5,0.5,0.5],np.float32), np.array([0.5,0.5,0.5],np.float32), 256
def load_frame(path):
    img = Image.open(path).convert("RGB").resize((RES,RES), Image.BILINEAR)
    arr = (np.asarray(img,np.float32)/255.0 - MEAN)/STD
    return torch.from_numpy(np.transpose(arr,(2,0,1)))

mani = pd.read_parquet(f"{REPO}/data/ffpp_manifest.parquet")
test = mani[mani["split"]=="test"] if "split" in mani.columns else mani
reals = test[test.label==0].sample(n=min(100,(test.label==0).sum()), random_state=42)
fakes = test[test.label==1].sample(n=min(100,(test.label==1).sum()), random_state=42)
sample = pd.concat([reals,fakes]).reset_index(drop=True)
print(f"sampled {len(sample)} ({(sample.label==0).sum()} real, {(sample.label==1).sum()} fake)")

probs, labels = [], []
with torch.no_grad():
    for i in range(0, len(sample), 32):
        rows = sample.iloc[i:i+32]
        imgs, labs = [], []
        for _, r in rows.iterrows():
            if os.path.exists(r["frame_path"]):
                imgs.append(load_frame(r["frame_path"])); labs.append(int(r["label"]))
        if not imgs: continue
        out = model({"image": torch.stack(imgs).to(device)}, inference=True)
        probs.extend(out["prob"].cpu().numpy().tolist()); labels.extend(labs)

probs, labels = np.array(probs), np.array(labels)
auc = roc_auc_score(labels, probs)
print(f"\n{'='*44}")
print(f"  IN-DOMAIN FF++ AUC: {auc:.4f}  (n={len(labels)})")
print(f"  fake-prob: [{probs.min():.3f},{probs.max():.3f}] mean {probs.mean():.3f}")
print(f"{'='*44}")
print("  PASS - foundation solid" if auc>=0.85 else
      "  MARGINAL - check preprocessing" if auc>=0.65 else
      "  FAIL - debug load/preprocessing")

AttributeError: 'NoneType' object has no attribute 'seek'. You can only torch.load from a file that is seekable. Please pre-load the data into a buffer like io.BytesIO and try to load from it instead.

In [13]:
import os, glob
import numpy as np, torch
from PIL import Image
import pandas as pd
from sklearn.metrics import roc_auc_score

REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
CKPT = f"{REPO}/weights/train_on_fs/xception.pth"

config = {"model_name":"xception","backbone_name":"xception",
    "backbone_config":{"mode":"original","num_classes":2,"inc":3,"dropout":False},
    "pretrained":None,"resolution":256,"loss_func":"cross_entropy",
    "mean":[0.5,0.5,0.5],"std":[0.5,0.5,0.5]}

# build model + load weights (strip module. prefix)
model = XceptionDetector(config)
sd = torch.load(CKPT, map_location="cpu")
new_sd = {(k[7:] if k.startswith("module.") else k): v for k,v in sd.items()}
missing, unexpected = model.load_state_dict(new_sd, strict=False)
print(f"loaded | missing:{len(missing)} unexpected:{len(unexpected)}")
if len(missing)>5: print("  WARN missing:", missing[:8])
if len(unexpected)>5: print("  WARN unexpected:", unexpected[:8])

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print("device:", device)

# preprocessing per config contract
MEAN, STD, RES = np.array([0.5,0.5,0.5],np.float32), np.array([0.5,0.5,0.5],np.float32), 256
def load_frame(path):
    img = Image.open(path).convert("RGB").resize((RES,RES), Image.BILINEAR)
    arr = (np.asarray(img,np.float32)/255.0 - MEAN)/STD
    return torch.from_numpy(np.transpose(arr,(2,0,1)))

# balanced FF++ test sample
mani = pd.read_parquet(f"{REPO}/data/ffpp_manifest.parquet")
test = mani[mani["split"]=="test"] if "split" in mani.columns else mani
reals = test[test.label==0].sample(n=min(100,(test.label==0).sum()), random_state=42)
fakes = test[test.label==1].sample(n=min(100,(test.label==1).sum()), random_state=42)
sample = pd.concat([reals,fakes]).reset_index(drop=True)
print(f"sampled {len(sample)} ({(sample.label==0).sum()} real, {(sample.label==1).sum()} fake)")

# inference
probs, labels = [], []
with torch.no_grad():
    for i in range(0, len(sample), 32):
        rows = sample.iloc[i:i+32]
        imgs, labs = [], []
        for _, r in rows.iterrows():
            if os.path.exists(r["frame_path"]):
                imgs.append(load_frame(r["frame_path"])); labs.append(int(r["label"]))
        if not imgs: continue
        out = model({"image": torch.stack(imgs).to(device)}, inference=True)
        probs.extend(out["prob"].cpu().numpy().tolist()); labels.extend(labs)

probs, labels = np.array(probs), np.array(labels)
auc = roc_auc_score(labels, probs)
print(f"\n{'='*44}")
print(f"  IN-DOMAIN FF++ AUC: {auc:.4f}  (n={len(labels)})")
print(f"  fake-prob: [{probs.min():.3f},{probs.max():.3f}] mean {probs.mean():.3f}")
print(f"{'='*44}")
print("  PASS - foundation solid" if auc>=0.85 else
      "  MARGINAL - check preprocessing" if auc>=0.65 else
      "  FAIL - debug load/preprocessing")

AttributeError: 'NoneType' object has no attribute 'seek'. You can only torch.load from a file that is seekable. Please pre-load the data into a buffer like io.BytesIO and try to load from it instead.

In [12]:
import shutil, os, io, torch
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
CKPT_DRIVE = f"{REPO}/weights/train_on_fs/xception.pth"
CKPT_LOCAL = "/content/xception.pth"

# copy to local runtime disk (off the flaky Drive mount)
shutil.copy(CKPT_DRIVE, CKPT_LOCAL)
print("copied to local:", os.path.getsize(CKPT_LOCAL)/1e6, "MB")

# load from LOCAL disk — no Drive seek issues
sd = torch.load(CKPT_LOCAL, map_location="cpu")
print("loaded OK, keys:", len(sd))
print("sample key:", list(sd.keys())[0])

copied to local: 87.767115 MB
loaded OK, keys: 283
sample key: module.backbone.conv1.weight


In [14]:
import os, glob
import numpy as np, torch
from PIL import Image
import pandas as pd
from sklearn.metrics import roc_auc_score

REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
CKPT = f"{REPO}/weights/train_on_fs/xception.pth"

config = {"model_name":"xception","backbone_name":"xception",
    "backbone_config":{"mode":"original","num_classes":2,"inc":3,"dropout":False},
    "pretrained":None,"resolution":256,"loss_func":"cross_entropy",
    "mean":[0.5,0.5,0.5],"std":[0.5,0.5,0.5]}

# build model + load weights (strip module. prefix)
model = XceptionDetector(config)
sd = torch.load(CKPT, map_location="cpu")
new_sd = {(k[7:] if k.startswith("module.") else k): v for k,v in sd.items()}
missing, unexpected = model.load_state_dict(new_sd, strict=False)
print(f"loaded | missing:{len(missing)} unexpected:{len(unexpected)}")
if len(missing)>5: print("  WARN missing:", missing[:8])
if len(unexpected)>5: print("  WARN unexpected:", unexpected[:8])

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print("device:", device)

# preprocessing per config contract
MEAN, STD, RES = np.array([0.5,0.5,0.5],np.float32), np.array([0.5,0.5,0.5],np.float32), 256
def load_frame(path):
    img = Image.open(path).convert("RGB").resize((RES,RES), Image.BILINEAR)
    arr = (np.asarray(img,np.float32)/255.0 - MEAN)/STD
    return torch.from_numpy(np.transpose(arr,(2,0,1)))

# balanced FF++ test sample
mani = pd.read_parquet(f"{REPO}/data/ffpp_manifest.parquet")
test = mani[mani["split"]=="test"] if "split" in mani.columns else mani
reals = test[test.label==0].sample(n=min(100,(test.label==0).sum()), random_state=42)
fakes = test[test.label==1].sample(n=min(100,(test.label==1).sum()), random_state=42)
sample = pd.concat([reals,fakes]).reset_index(drop=True)
print(f"sampled {len(sample)} ({(sample.label==0).sum()} real, {(sample.label==1).sum()} fake)")

# inference
probs, labels = [], []
with torch.no_grad():
    for i in range(0, len(sample), 32):
        rows = sample.iloc[i:i+32]
        imgs, labs = [], []
        for _, r in rows.iterrows():
            if os.path.exists(r["frame_path"]):
                imgs.append(load_frame(r["frame_path"])); labs.append(int(r["label"]))
        if not imgs: continue
        out = model({"image": torch.stack(imgs).to(device)}, inference=True)
        probs.extend(out["prob"].cpu().numpy().tolist()); labels.extend(labs)

probs, labels = np.array(probs), np.array(labels)
auc = roc_auc_score(labels, probs)
print(f"\n{'='*44}")
print(f"  IN-DOMAIN FF++ AUC: {auc:.4f}  (n={len(labels)})")
print(f"  fake-prob: [{probs.min():.3f},{probs.max():.3f}] mean {probs.mean():.3f}")
print(f"{'='*44}")
print("  PASS - foundation solid" if auc>=0.85 else
      "  MARGINAL - check preprocessing" if auc>=0.65 else
      "  FAIL - debug load/preprocessing")

AttributeError: 'NoneType' object has no attribute 'seek'. You can only torch.load from a file that is seekable. Please pre-load the data into a buffer like io.BytesIO and try to load from it instead.

In [15]:
import os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
xd = f"{REPO}/external/DeepfakeBench/training/detectors/xception_detector.py"
src = open(xd).read()
bb = src[src.find("def build_backbone"):src.find("def build_loss")]
print(bb)

def build_backbone(self, config):
        # prepare the backbone
        backbone_class = BACKBONE[config['backbone_name']]
        model_config = config['backbone_config']
        backbone = backbone_class(model_config)
        # if donot load the pretrained weights, fail to get good results
        state_dict = torch.load(config['pretrained'])
        for name, weights in state_dict.items():
            if 'pointwise' in name:
                state_dict[name] = weights.unsqueeze(-1).unsqueeze(-1)
        state_dict = {k:v for k, v in state_dict.items() if 'fc' not in k}
        backbone.load_state_dict(state_dict, False)
        logger.info('Load pretrained model successfully!')
        return backbone
    
    


In [16]:
import os, glob
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
pre = f"{REPO}/external/DeepfakeBench/training/pretrained/xception-b5690688.pth"
print("imagenet xception exists:", os.path.exists(pre))
print("pretrained folder contents:", glob.glob(f"{REPO}/external/DeepfakeBench/training/pretrained/*"))

imagenet xception exists: False
pretrained folder contents: ['/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/external/DeepfakeBench/training/pretrained/readme.md']


In [17]:
import os, glob
import numpy as np, torch
from PIL import Image
import pandas as pd
from sklearn.metrics import roc_auc_score

REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
CKPT_LOCAL = "/content/xception.pth"

config = {"model_name":"xception","backbone_name":"xception",
    "backbone_config":{"mode":"original","num_classes":2,"inc":3,"dropout":False},
    "pretrained":None,"resolution":256,"loss_func":"cross_entropy",
    "mean":[0.5,0.5,0.5],"std":[0.5,0.5,0.5]}

# --- bypass build_backbone's pretrained load (we load full trained weights next) ---
_orig_load = torch.load
def _patched_load(f, *a, **k):
    if f is None:                      # build_backbone calls torch.load(None)
        return {}                      # -> empty dict, load becomes no-op
    return _orig_load(f, *a, **k)
torch.load = _patched_load

model = XceptionDetector(config)       # builds with empty pretrained (fine)

torch.load = _orig_load                # restore for the real load

# --- load YOUR trained weights (the ones that matter) ---
sd = torch.load(CKPT_LOCAL, map_location="cpu")
new_sd = {(k[7:] if k.startswith("module.") else k): v for k,v in sd.items()}
missing, unexpected = model.load_state_dict(new_sd, strict=False)
print(f"loaded | missing:{len(missing)} unexpected:{len(unexpected)}")
if len(missing)>5: print("  missing sample:", missing[:8])
if len(unexpected)>5: print("  unexpected sample:", unexpected[:8])

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print("device:", device)

MEAN, STD, RES = np.array([0.5,0.5,0.5],np.float32), np.array([0.5,0.5,0.5],np.float32), 256
def load_frame(path):
    img = Image.open(path).convert("RGB").resize((RES,RES), Image.BILINEAR)
    arr = (np.asarray(img,np.float32)/255.0 - MEAN)/STD
    return torch.from_numpy(np.transpose(arr,(2,0,1)))

mani = pd.read_parquet(f"{REPO}/data/ffpp_manifest.parquet")
test = mani[mani["split"]=="test"] if "split" in mani.columns else mani
reals = test[test.label==0].sample(n=min(100,(test.label==0).sum()), random_state=42)
fakes = test[test.label==1].sample(n=min(100,(test.label==1).sum()), random_state=42)
sample = pd.concat([reals,fakes]).reset_index(drop=True)
print(f"sampled {len(sample)} ({(sample.label==0).sum()} real, {(sample.label==1).sum()} fake)")

probs, labels = [], []
with torch.no_grad():
    for i in range(0, len(sample), 32):
        rows = sample.iloc[i:i+32]
        imgs, labs = [], []
        for _, r in rows.iterrows():
            if os.path.exists(r["frame_path"]):
                imgs.append(load_frame(r["frame_path"])); labs.append(int(r["label"]))
        if not imgs: continue
        out = model({"image": torch.stack(imgs).to(device)}, inference=True)
        probs.extend(out["prob"].cpu().numpy().tolist()); labels.extend(labs)

probs, labels = np.array(probs), np.array(labels)
auc = roc_auc_score(labels, probs)
print(f"\n{'='*44}\n  IN-DOMAIN FF++ AUC: {auc:.4f}  (n={len(labels)})")
print(f"  fake-prob: [{probs.min():.3f},{probs.max():.3f}] mean {probs.mean():.3f}\n{'='*44}")
print("  PASS - foundation solid" if auc>=0.85 else "  MARGINAL - check preprocessing" if auc>=0.65 else "  FAIL - debug")

loaded | missing:0 unexpected:0
device: cuda
sampled 200 (100 real, 100 fake)

  IN-DOMAIN FF++ AUC: 0.7957  (n=200)
  fake-prob: [0.000,1.000] mean 0.254
  MARGINAL - check preprocessing


In [18]:
import numpy as np, torch, os
from PIL import Image
from sklearn.metrics import roc_auc_score
import pandas as pd

REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
MEAN, STD, RES = np.array([0.5,0.5,0.5],np.float32), np.array([0.5,0.5,0.5],np.float32), 256
def load_frame(path):
    img = Image.open(path).convert("RGB").resize((RES,RES), Image.BILINEAR)
    arr = (np.asarray(img,np.float32)/255.0 - MEAN)/STD
    return torch.from_numpy(np.transpose(arr,(2,0,1)))

mani = pd.read_parquet(f"{REPO}/data/ffpp_manifest.parquet")
test = mani[mani["split"]=="test"] if "split" in mani.columns else mani
device = "cuda"

def auc_for_method(method, n=80):
    reals = test[test.label==0].sample(n=n, random_state=1)
    fakes = test[(test.label==1) & (test.method==method)].sample(n=min(n,(((test.label==1)&(test.method==method)).sum())), random_state=1)
    s = pd.concat([reals,fakes]).reset_index(drop=True)
    probs, labs = [], []
    with torch.no_grad():
        for i in range(0,len(s),32):
            rows=s.iloc[i:i+32]; imgs,l=[],[]
            for _,r in rows.iterrows():
                if os.path.exists(r["frame_path"]): imgs.append(load_frame(r["frame_path"])); l.append(int(r["label"]))
            if not imgs: continue
            out=model({"image":torch.stack(imgs).to(device)}, inference=True)
            probs.extend(out["prob"].cpu().numpy().tolist()); labs.extend(l)
    return roc_auc_score(labs,probs), len(labs)

print("AUC per FF++ method (train_on_fs = face-swap-trained checkpoint):")
for m in ["faceswap","deepfakes","face2face","neuraltextures"]:
    a,n = auc_for_method(m)
    print(f"  {m:16s}: AUC {a:.4f}  (n={n})")

AUC per FF++ method (train_on_fs = face-swap-trained checkpoint):
  faceswap        : AUC 0.9667  (n=160)
  deepfakes       : AUC 0.7528  (n=160)
  face2face       : AUC 0.6341  (n=160)
  neuraltextures  : AUC 0.6094  (n=160)


In [21]:
import os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
print("inference.py in src:", os.path.exists(f"{REPO}/src/inference.py"))

inference.py in src: True


In [20]:
import glob, os
# search everywhere for the uploaded inference.py
hits = glob.glob("/content/**/inference.py", recursive=True)
print("found at:")
for h in hits:
    print(f"  {h}  ({os.path.getsize(h)} bytes)")

KeyboardInterrupt: 

In [22]:
import sys, importlib
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
sys.path.insert(0, f"{REPO}/src")
import inference; importlib.reload(inference)
import pandas as pd

# 1. load detector
model, device, load_info = inference.load_detector(
    dfb_root=f"{REPO}/external/DeepfakeBench",
    backbone_name="xception",
    ckpt_path=f"{REPO}/weights/train_on_fs/xception.pth",
)
print("load:", load_info, "| device:", device)

# 2. score FF++ test split
mani = pd.read_parquet(f"{REPO}/data/ffpp_manifest.parquet")
scores = inference.score_manifest(model, device, mani, splits=["test"], batch_size=64)
print(f"\nscored {len(scores)} frames")

# 3. per-method AUC
aucs = inference.quick_auc(scores)
print("\nAUC per method:")
for k, v in aucs.items(): print(f"  {k:16s}: {v:.4f}")

# 4. save
import os
os.makedirs(f"{REPO}/reports/scores", exist_ok=True)
out = f"{REPO}/reports/scores/xception_ffpp_test.parquet"
scores.to_parquet(out, index=False)
print(f"\nsaved -> {out} ({os.path.getsize(out)/1e6:.1f} MB)")

load: {'missing': 0, 'unexpected': 0} | device: cuda
  scored 64/22388
  scored 704/22388
  scored 1344/22388
  scored 1984/22388
  scored 2624/22388
  scored 3264/22388
  scored 3904/22388
  scored 4544/22388
  scored 5184/22388
  scored 5824/22388
  scored 6464/22388
  scored 7104/22388
  scored 7744/22388
  scored 8384/22388
  scored 9024/22388
  scored 9664/22388
  scored 10304/22388
  scored 10944/22388
  scored 11584/22388
  scored 12224/22388
  scored 12864/22388
  scored 13504/22388
  scored 14144/22388
  scored 14784/22388
  scored 15424/22388
  scored 16064/22388
  scored 16704/22388
  scored 17344/22388
  scored 17984/22388
  scored 18624/22388
  scored 19264/22388
  scored 19904/22388
  scored 20544/22388
  scored 21184/22388
  scored 21824/22388

scored 22388 frames

AUC per method:
  overall         : 0.7526

saved -> /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/reports/scores/xception_ffpp_test.parquet (0.8 MB)


In [23]:
import pandas as pd
from sklearn.metrics import roc_auc_score
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
scores = pd.read_parquet(f"{REPO}/reports/scores/xception_ffpp_test.parquet")

print("rows:", len(scores))
print("methods present:", scores['method'].unique())
print("label balance:", scores['label'].value_counts().to_dict())

print("\n=== AUC per method (real vs each fake method) ===")
reals = scores[scores.label==0]
for m in ['faceswap','deepfakes','face2face','neuraltextures']:
    fakes_m = scores[(scores.label==1) & (scores.method==m)]
    if len(fakes_m) == 0:
        print(f"  {m}: no fake frames"); continue
    sub = pd.concat([reals, fakes_m])
    auc = roc_auc_score(sub.label, sub.prob_fake)
    print(f"  {m:16s}: AUC {auc:.4f}  (real={len(reals)}, fake={len(fakes_m)})")

# overall (all methods pooled)
print(f"\noverall pooled AUC: {roc_auc_score(scores.label, scores.prob_fake):.4f}")

rows: 22388
methods present: ['real' 'deepfakes' 'face2face' 'faceswap' 'neuraltextures']
label balance: {1: 17909, 0: 4479}

=== AUC per method (real vs each fake method) ===
  faceswap        : AUC 0.9665  (real=4479, fake=4477)
  deepfakes       : AUC 0.8085  (real=4479, fake=4473)
  face2face       : AUC 0.6341  (real=4479, fake=4480)
  neuraltextures  : AUC 0.6015  (real=4479, fake=4479)

overall pooled AUC: 0.7526


In [24]:
import sys, os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
sys.path.insert(0, f"{REPO}/src")

# what's in calibration.py?
import inspect
print("=== calibration.py functions ===")
cal_src = open(f"{REPO}/src/calibration.py").read()
import re
for m in re.finditer(r"^(def|class)\s+(\w+)", cal_src, re.M):
    print(f"  {m.group(1)} {m.group(2)}")
print("\n=== metrics.py functions ===")
met_src = open(f"{REPO}/src/metrics.py").read()
for m in re.finditer(r"^(def|class)\s+(\w+)", met_src, re.M):
    print(f"  {m.group(1)} {m.group(2)}")

=== calibration.py functions ===
  def leakage_safe_split
  def assert_three_way_disjoint
  class _BaseCalibrator
  class Uncalibrated
  class PlattScaling
  class IsotonicCalibration
  class TemperatureScaling
  class BetaCalibration
  class HistogramBinning
  class BBQCalibration
  class HybridCalibrator
  def get_calibrator
  def fit_predict

=== metrics.py functions ===
  def _check
  def logit
  def sigmoid
  def _bin_indices
  def ece
  def mce
  def ece_debiased
  def ece_sweep
  def reliability_curve
  def brier_score
  def nll
  def roc_auc
  def bootstrap_ci
  def _midrank
  def _fast_delong
  def _compute_ground_truth_statistics
  def delong_roc_variance
  def delong_roc_test


In [27]:
import sys
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB_TRAIN = f"{REPO}/external/DeepfakeBench/training"

# remove DeepfakeBench training dir from path (it has a conflicting 'metrics' package)
sys.path = [p for p in sys.path if p != DFB_TRAIN]
# also drop any cached conflicting modules
for mod in ["metrics", "calibration"]:
    if mod in sys.modules:
        # only drop the DeepfakeBench metrics, keep nothing stale
        del sys.modules[mod]
# clear DeepfakeBench's metrics submodules too
for k in list(sys.modules.keys()):
    if k.startswith("metrics.") or k == "metrics":
        del sys.modules[k]

# now put src FIRST
if f"{REPO}/src" in sys.path:
    sys.path.remove(f"{REPO}/src")
sys.path.insert(0, f"{REPO}/src")

# import YOUR modules
import metrics, calibration
print("metrics from:", metrics.__file__)
print("calibration from:", calibration.__file__)
print("has logit:", hasattr(metrics, "logit"))

metrics from: /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/src/metrics.py
calibration from: /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/src/calibration.py
has logit: True


In [28]:
import inspect
import calibration, metrics

for fn in [calibration.leakage_safe_split, calibration.get_calibrator, calibration.fit_predict]:
    print(f"calibration.{fn.__name__}{inspect.signature(fn)}")
print()
for fn in [metrics.ece, metrics.ece_debiased, metrics.brier_score, metrics.bootstrap_ci]:
    print(f"metrics.{fn.__name__}{inspect.signature(fn)}")
print()
print("HybridCalibrator.__init__", inspect.signature(calibration.HybridCalibrator.__init__))
print("\n=== fit_predict source ===")
print(inspect.getsource(calibration.fit_predict))
print("\n=== leakage_safe_split source ===")
print(inspect.getsource(calibration.leakage_safe_split))

calibration.leakage_safe_split(y, groups=None, calib_frac=0.5, seed=42)
calibration.get_calibrator(name, **kwargs)
calibration.fit_predict(name, p_calib, y_calib, p_eval, **kwargs)

metrics.ece(p, y, n_bins=15, scheme='equal_mass')
metrics.ece_debiased(p, y, n_bins=15, scheme='equal_mass')
metrics.brier_score(p, y)
metrics.bootstrap_ci(fn, arrays, B=2000, ci=(2.5, 97.5), seed=42, stratify_by=None)

HybridCalibrator.__init__ (self, switch_threshold_n=1000)

=== fit_predict source ===
def fit_predict(name, p_calib, y_calib, p_eval, **kwargs):
    """Convenience: fit a named calibrator on the calibration split, apply to the eval split."""
    cal = get_calibrator(name, **kwargs).fit(p_calib, y_calib)
    out = cal.predict(p_eval)
    return out, cal


=== leakage_safe_split source ===
def leakage_safe_split(y, groups=None, calib_frac=0.5, seed=42):
    """
    Split a held-out pool into (calib_idx, test_idx).

    groups : optional identity/video id per row. When supplied the split is GRO

In [29]:
import inspect, calibration, metrics
print("ece:", inspect.signature(metrics.ece))
print("brier_score:", inspect.signature(metrics.brier_score))
print("get_calibrator:", inspect.signature(calibration.get_calibrator))
# what names does get_calibrator accept?
print("\nget_calibrator source:")
print(inspect.getsource(calibration.get_calibrator))

ece: (p, y, n_bins=15, scheme='equal_mass')
brier_score: (p, y)
get_calibrator: (name, **kwargs)

get_calibrator source:
def get_calibrator(name, **kwargs):
    """Factory. e.g. get_calibrator('hybrid', switch_threshold_n=1000)."""
    key = name.lower()
    if key not in _REGISTRY:
        raise KeyError(f"unknown calibrator '{name}'. available: {AVAILABLE_METHODS}")
    return _REGISTRY[key](**kwargs)



In [40]:
import glob, os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
print("in src/:", os.path.exists(f"{REPO}/src/calibrate_scores.py"))
# search everywhere
hits = glob.glob("/content/**/calibrate_scores.py", recursive=True)
print("found at:", hits)

in src/: True
found at: ['/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/src/calibrate_scores.py']


In [45]:
import sys, os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
SRC = f"{REPO}/src"

# is the file really there?
print("file exists:", os.path.exists(f"{SRC}/calibrate_scores.py"))
# is src on the path?
print("src in sys.path:", SRC in sys.path)
# show first 5 path entries
print("path[:5]:", sys.path[:5])

# force it cleanly
if SRC not in sys.path:
    sys.path.insert(0, SRC)
else:
    # move it to front
    sys.path.remove(SRC); sys.path.insert(0, SRC)

# clear any cached version
for m in ["calibrate_scores"]:
    sys.modules.pop(m, None)

# try direct import
try:
    import calibrate_scores
    print("IMPORT OK:", calibrate_scores.__file__)
except Exception as e:
    print("import failed:", type(e).__name__, e)
    # fallback: load directly by file path
    import importlib.util
    spec = importlib.util.spec_from_file_location("calibrate_scores", f"{SRC}/calibrate_scores.py")
    calibrate_scores = importlib.util.module_from_spec(spec)
    sys.modules["calibrate_scores"] = calibrate_scores
    spec.loader.exec_module(calibrate_scores)
    print("loaded via direct file path:", calibrate_scores.__file__)

file exists: True
src in sys.path: True
path[:5]: ['/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/src', '/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/src', '/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/src', '/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/src', '/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/external/DeepfakeBench']
import failed: ModuleNotFoundError No module named 'calibrate_scores'
loaded via direct file path: /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/src/calibrate_scores.py


In [46]:
import os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# calibrate_scores is already loaded in memory from the previous cell
results = calibrate_scores.run_calibration_experiment(
    scores_path=f"{REPO}/reports/scores/xception_ffpp_test.parquet",
    in_domain_method="faceswap",
    B=2000,
)

os.makedirs(f"{REPO}/reports/calibration", exist_ok=True)
results["pooled"].to_csv(f"{REPO}/reports/calibration/xception_ffpp_pooled.csv", index=False)
results["coupling"].to_csv(f"{REPO}/reports/calibration/xception_ffpp_coupling.csv", index=False)
print("\nsaved result tables to reports/calibration/")

=== leakage-safe split (by identity) ===
  mode=group_disjoint  n_calib=11198  n_test=11190
  calib_pos_rate=0.800  test_pos_rate=0.800

=== R1/R2: pooled calibration (FF++ all methods, test split) ===


ValueError: too many values to unpack (expected 2)

In [47]:
import metrics as met
import numpy as np, inspect
# what does bootstrap_ci return?
print(inspect.getsource(met.bootstrap_ci).split("return")[-1][:300])
# quick live test
p = np.array([0.1,0.9,0.3,0.7,0.2,0.8]*20)
y = np.array([0,1,0,1,0,1]*20)
out = met.bootstrap_ci(lambda a,b: met.ece(a,b,n_bins=5), (p,y), B=100)
print("\nreturn type:", type(out), "| value:", out)

 point, float(lo), float(hi)


return type: <class 'tuple'> | value: (0.14, 0.13583333333333336, 0.19337499999999996)


In [48]:
import re
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
path = f"{REPO}/src/calibrate_scores.py"
src = open(path).read()
src = src.replace(
    "    lo, hi = met.bootstrap_ci(_ece_fn, (p, y), B=B)",
    "    _pt, lo, hi = met.bootstrap_ci(_ece_fn, (p, y), B=B)"
)
open(path, "w").write(src)
print("patched _summary unpacking")

# reload via direct file path (the import quirk from before)
import importlib.util, sys
spec = importlib.util.spec_from_file_location("calibrate_scores", path)
calibrate_scores = importlib.util.module_from_spec(spec)
sys.modules["calibrate_scores"] = calibrate_scores
spec.loader.exec_module(calibrate_scores)
print("reloaded")

patched _summary unpacking
reloaded


In [49]:
import os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
results = calibrate_scores.run_calibration_experiment(
    scores_path=f"{REPO}/reports/scores/xception_ffpp_test.parquet",
    in_domain_method="faceswap",
    B=2000,
)
os.makedirs(f"{REPO}/reports/calibration", exist_ok=True)
results["pooled"].to_csv(f"{REPO}/reports/calibration/xception_ffpp_pooled.csv", index=False)
results["coupling"].to_csv(f"{REPO}/reports/calibration/xception_ffpp_coupling.csv", index=False)
print("\nsaved result tables")

=== leakage-safe split (by identity) ===
  mode=group_disjoint  n_calib=11198  n_test=11190
  calib_pos_rate=0.800  test_pos_rate=0.800

=== R1/R2: pooled calibration (FF++ all methods, test split) ===
               set     n      ECE   ECE_lo   ECE_hi    Brier
raw (uncalibrated) 11190 0.455291 0.446980 0.464659 0.440240
 hybrid-calibrated 11190 0.047267 0.017096 0.028591 0.139922
  --> ECE reduced by 0.4080 (89.6%)

=== R3: forgery coupling (calibration fit on 'faceswap'+real, n_fit=4480) ===
  applied per-method to the TEST split:
    faceswap         n= 4476  AUC=0.974  ECE_raw=0.0437  ECE_cal=0.0364  <- in-domain
    deepfakes        n= 4474  AUC=0.839  ECE_raw=0.2379  ECE_cal=0.1762
    face2face        n= 4479  AUC=0.625  ECE_raw=0.4118  ECE_cal=0.3195
    neuraltextures   n= 4478  AUC=0.600  ECE_raw=0.4314  ECE_cal=0.3428

  INTERPRETATION: if ECE_cal stays low for the in-domain method but grows
  for reenactment methods, the calibration does NOT transfer across forgery
  types

In [50]:
import metrics as met, inspect
print(inspect.getsource(met.bootstrap_ci))

def bootstrap_ci(fn, arrays, B=2000, ci=(2.5, 97.5), seed=42, stratify_by=None):
    """
    Paired nonparametric bootstrap CI for any metric.

    fn          : callable taking the resampled arrays positionally, returning a scalar.
    arrays      : list/tuple of equal-length arrays passed to fn in order (e.g. [p, y]).
    stratify_by : optional label array; if given, resample within each class so rare
                  classes (U2R-style) do not vanish from a resample.

    Returns (point, lo, hi) where point is fn on the full data.
    """
    arrays = [np.asarray(a) for a in arrays]
    n = len(arrays[0])
    if any(len(a) != n for a in arrays):
        raise ValueError("all arrays must share length")
    rng = np.random.default_rng(seed)
    point = float(fn(*arrays))

    if stratify_by is not None:
        stratify_by = np.asarray(stratify_by).ravel()
        groups = [np.where(stratify_by == c)[0] for c in np.unique(stratify_by)]

    stats_ = np.empty(B, dtype=float)
    valid

In [51]:
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
path = f"{REPO}/src/calibrate_scores.py"
src = open(path).read()

# replace _summary to use bootstrap_ci's own point (consistent point+CI by construction)
old = '''def _summary(p, y, label, B=2000):
    """ECE + Brier with bootstrap CI on ECE."""
    e = met.ece(p, y, n_bins=15, scheme="equal_mass")
    b = met.brier_score(p, y)
    _pt, lo, hi = met.bootstrap_ci(_ece_fn, (p, y), B=B)
    return {"set": label, "n": len(y), "ECE": e, "ECE_lo": lo, "ECE_hi": hi, "Brier": b}'''

new = '''def _summary(p, y, label, B=2000):
    """ECE + Brier with bootstrap CI on ECE. Point comes from bootstrap_ci itself
    so point and CI are always mutually consistent."""
    b = met.brier_score(p, y)
    pt, lo, hi = met.bootstrap_ci(_ece_fn, (p, y), B=B, stratify_by=y)
    return {"set": label, "n": len(y), "ECE": pt, "ECE_lo": lo, "ECE_hi": hi, "Brier": b}'''

src = src.replace(old, new)
open(path, "w").write(src)
print("patched _summary (consistent point+CI, label-stratified bootstrap)")

# reload
import importlib.util, sys
spec = importlib.util.spec_from_file_location("calibrate_scores", path)
calibrate_scores = importlib.util.module_from_spec(spec)
sys.modules["calibrate_scores"] = calibrate_scores
spec.loader.exec_module(calibrate_scores)
print("reloaded")

patched _summary (consistent point+CI, label-stratified bootstrap)
reloaded


In [52]:
import os
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
results = calibrate_scores.run_calibration_experiment(
    scores_path=f"{REPO}/reports/scores/xception_ffpp_test.parquet",
    in_domain_method="faceswap",
    B=2000,
)
os.makedirs(f"{REPO}/reports/calibration", exist_ok=True)
results["pooled"].to_csv(f"{REPO}/reports/calibration/xception_ffpp_pooled.csv", index=False)
results["coupling"].to_csv(f"{REPO}/reports/calibration/xception_ffpp_coupling.csv", index=False)
print("\nsaved result tables")

=== leakage-safe split (by identity) ===
  mode=group_disjoint  n_calib=11198  n_test=11190
  calib_pos_rate=0.800  test_pos_rate=0.800

=== R1/R2: pooled calibration (FF++ all methods, test split) ===
               set     n      ECE   ECE_lo   ECE_hi    Brier
raw (uncalibrated) 11190 0.455291 0.448097 0.462982 0.440240
 hybrid-calibrated 11190 0.047267 0.040245 0.060783 0.139922
  --> ECE reduced by 0.4080 (89.6%)

=== R3: forgery coupling (calibration fit on 'faceswap'+real, n_fit=4480) ===
  applied per-method to the TEST split:
    faceswap         n= 4476  AUC=0.974  ECE_raw=0.0437  ECE_cal=0.0364  <- in-domain
    deepfakes        n= 4474  AUC=0.839  ECE_raw=0.2379  ECE_cal=0.1762
    face2face        n= 4479  AUC=0.625  ECE_raw=0.4118  ECE_cal=0.3195
    neuraltextures   n= 4478  AUC=0.600  ECE_raw=0.4314  ECE_cal=0.3428

  INTERPRETATION: if ECE_cal stays low for the in-domain method but grows
  for reenactment methods, the calibration does NOT transfer across forgery
  types

In [53]:
import sys, os, shutil
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# copy the reenactment-trained xception to local (off Drive, for reliable torch.load)
src_ckpt = f"{REPO}/weights/train_on_fr/xception.pth"
# wait - is train_on_fr/xception.pth in the repo weights, or only in the shortcut?
print("in repo weights:", os.path.exists(src_ckpt))
if not os.path.exists(src_ckpt):
    # copy from the shortcut source
    sc = "/content/drive/MyDrive/CDTS_Research/df40_weights/train_on_fr/xception.pth"
    print("in shortcut:", os.path.exists(sc))
    if os.path.exists(sc):
        os.makedirs(f"{REPO}/weights/train_on_fr", exist_ok=True)
        shutil.copy(sc, src_ckpt)
        print("copied train_on_fr/xception.pth to repo weights")
print("checkpoint ready:", os.path.exists(src_ckpt), f"({os.path.getsize(src_ckpt)/1e6:.0f}MB)" if os.path.exists(src_ckpt) else "")

in repo weights: False
in shortcut: True
copied train_on_fr/xception.pth to repo weights
checkpoint ready: True (88MB)


In [55]:
import sys, importlib
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB_TRAIN = f"{REPO}/external/DeepfakeBench/training"

# 1) purge any cached metrics/calibration/inference so they reload cleanly
for k in list(sys.modules.keys()):
    if k == "metrics" or k.startswith("metrics.") or k in ("calibration","inference","calibrate_scores"):
        del sys.modules[k]

# 2) put DeepfakeBench training FIRST (so its 'metrics' package wins for inference)
sys.path = [p for p in sys.path if p not in (DFB_TRAIN, f"{REPO}/src")]
sys.path.insert(0, f"{REPO}/external/DeepfakeBench")
sys.path.insert(0, DFB_TRAIN)        # DeepfakeBench metrics package — first
sys.path.append(f"{REPO}/src")       # src LAST (inference.py findable, but DFB metrics wins)

# 3) verify which metrics resolves now
import metrics as _m
print("metrics resolves to:", getattr(_m, "__file__", "package:"+str(_m.__path__) if hasattr(_m,"__path__") else "?"))

# 4) load inference.py by direct file path (so src ordering doesn't matter for IT)
import importlib.util
spec = importlib.util.spec_from_file_location("inference", f"{REPO}/src/inference.py")
inference = importlib.util.module_from_spec(spec)
sys.modules["inference"] = inference
spec.loader.exec_module(inference)
print("inference loaded")

# 5) now load the detector
import pandas as pd
model, device, load_info = inference.load_detector(
    dfb_root=f"{REPO}/external/DeepfakeBench",
    backbone_name="xception",
    ckpt_path=f"{REPO}/weights/train_on_fr/xception.pth",
)
print("detector load:", load_info, "| device:", device)

metrics resolves to: /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/external/DeepfakeBench/training/metrics/__init__.py
inference loaded
detector load: {'missing': 0, 'unexpected': 0} | device: cuda


In [56]:
import pandas as pd, os
from sklearn.metrics import roc_auc_score
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

mani = pd.read_parquet(f"{REPO}/data/ffpp_manifest.parquet")
scores = inference.score_manifest(model, device, mani, splits=["test"], batch_size=64)
print(f"scored {len(scores)} frames")
scores.to_parquet(f"{REPO}/reports/scores/xception_FR_ffpp_test.parquet", index=False)

reals = scores[scores.label==0]
print("\n=== AUC per method (REENACTMENT-trained checkpoint) ===")
for m in ['faceswap','deepfakes','face2face','neuraltextures']:
    fk = scores[(scores.label==1)&(scores.method==m)]
    sub = pd.concat([reals, fk])
    print(f"  {m:16s}: AUC {roc_auc_score(sub.label, sub.prob_fake):.4f}")

  scored 64/22388
  scored 704/22388
  scored 1344/22388
  scored 1984/22388
  scored 2624/22388
  scored 3264/22388
  scored 3904/22388
  scored 4544/22388
  scored 5184/22388
  scored 5824/22388
  scored 6464/22388
  scored 7104/22388
  scored 7744/22388
  scored 8384/22388
  scored 9024/22388
  scored 9664/22388
  scored 10304/22388
  scored 10944/22388
  scored 11584/22388
  scored 12224/22388
  scored 12864/22388
  scored 13504/22388
  scored 14144/22388
  scored 14784/22388
  scored 15424/22388
  scored 16064/22388
  scored 16704/22388
  scored 17344/22388
  scored 17984/22388
  scored 18624/22388
  scored 19264/22388
  scored 19904/22388
  scored 20544/22388
  scored 21184/22388
  scored 21824/22388
scored 22388 frames

=== AUC per method (REENACTMENT-trained checkpoint) ===
  faceswap        : AUC 0.9665
  deepfakes       : AUC 0.8085
  face2face       : AUC 0.6341
  neuraltextures  : AUC 0.6015


In [57]:
import torch, io
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# load both checkpoints, compare a weight tensor
def load_local(path):
    import shutil, os
    local = f"/content/_cmp_{os.path.basename(os.path.dirname(path))}.pth"
    shutil.copy(path, local)
    return torch.load(local, map_location="cpu")

fs = load_local(f"{REPO}/weights/train_on_fs/xception.pth")
fr = load_local(f"{REPO}/weights/train_on_fr/xception.pth")

# pick a representative weight and compare
key = "module.backbone.conv1.weight"
import numpy as np
same = torch.allclose(fs[key], fr[key])
diff = (fs[key] - fr[key]).abs().max().item()
print(f"conv1.weight identical: {same}, max abs diff: {diff:.6f}")

# check the final classifier layer (most likely to differ between training sets)
fc_keys = [k for k in fs.keys() if 'last' in k.lower() or 'fc' in k.lower() or 'classifier' in k.lower()]
print("classifier-like keys:", fc_keys[:5])
if fc_keys:
    k = fc_keys[0]
    print(f"{k} identical: {torch.allclose(fs[k], fr[k])}, max diff: {(fs[k]-fr[k]).abs().max().item():.6f}")

# also: what's currently loaded in the model? check one param
mp = dict(model.named_parameters())
mkey = [k for k in mp if 'conv1' in k][0]
print(f"\nmodel's {mkey} matches FR conv1:", torch.allclose(mp[mkey].cpu(), fr[key]))
print(f"model's {mkey} matches FS conv1:", torch.allclose(mp[mkey].cpu(), fs[key]))

conv1.weight identical: False, max abs diff: 0.107591
classifier-like keys: ['module.backbone.last_linear.weight', 'module.backbone.last_linear.bias']
module.backbone.last_linear.weight identical: False, max diff: 0.019500

model's backbone.conv1.weight matches FR conv1: False
model's backbone.conv1.weight matches FS conv1: True


In [58]:
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
path = f"{REPO}/src/inference.py"
src = open(path).read()

old = '''    # copy checkpoint to local disk if it's on Drive (Drive FUSE mount has
    # intermittent torch.load seek failures; local read is reliable)
    if ckpt_path.startswith("/content/drive"):
        local = f"/content/_ckpt_{backbone_name}.pth"
        if not os.path.exists(local) or os.path.getsize(local) != os.path.getsize(ckpt_path):
            shutil.copy(ckpt_path, local)
        ckpt_path = local'''

new = '''    # copy checkpoint to local disk if it's on Drive (Drive FUSE mount has
    # intermittent torch.load seek failures; local read is reliable).
    # Use a path-hash in the local name so DIFFERENT checkpoints of the same
    # backbone (e.g. train_on_fs vs train_on_fr, both 88MB) never collide.
    if ckpt_path.startswith("/content/drive"):
        import hashlib
        tag = hashlib.md5(ckpt_path.encode()).hexdigest()[:8]
        local = f"/content/_ckpt_{backbone_name}_{tag}.pth"
        if not os.path.exists(local) or os.path.getsize(local) != os.path.getsize(ckpt_path):
            shutil.copy(ckpt_path, local)
        ckpt_path = local'''

src = src.replace(old, new)
open(path, "w").write(src)
print("patched: local checkpoint name now includes path-hash (no collision)")

# clear the stale local copies
import glob, os
for f in glob.glob("/content/_ckpt_*.pth") + glob.glob("/content/_cmp_*.pth"):
    os.remove(f)
print("cleared stale local checkpoints")

patched: local checkpoint name now includes path-hash (no collision)
cleared stale local checkpoints


In [59]:
import sys, importlib.util, hashlib, torch
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# reload patched inference
spec = importlib.util.spec_from_file_location("inference", f"{REPO}/src/inference.py")
inference = importlib.util.module_from_spec(spec)
sys.modules["inference"] = inference
spec.loader.exec_module(inference)

# rebuild model with FR checkpoint
model, device, load_info = inference.load_detector(
    dfb_root=f"{REPO}/external/DeepfakeBench",
    backbone_name="xception",
    ckpt_path=f"{REPO}/weights/train_on_fr/xception.pth",
)
print("load:", load_info, "device:", device)

# VERIFY model actually has FR weights now
fr_path = f"{REPO}/weights/train_on_fr/xception.pth"
tag = hashlib.md5(fr_path.encode()).hexdigest()[:8]
fr = torch.load(f"/content/_ckpt_xception_{tag}.pth", map_location="cpu")
mp = dict(model.named_parameters())
mkey = [k for k in mp if 'conv1' in k][0]
print("model matches FR conv1:", torch.allclose(mp[mkey].cpu(), fr["module.backbone.conv1.weight"]))

load: {'missing': 0, 'unexpected': 0} device: cuda
model matches FR conv1: True


In [60]:
import pandas as pd, os
from sklearn.metrics import roc_auc_score
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

mani = pd.read_parquet(f"{REPO}/data/ffpp_manifest.parquet")
scores = inference.score_manifest(model, device, mani, splits=["test"], batch_size=64)
print(f"scored {len(scores)} frames")
scores.to_parquet(f"{REPO}/reports/scores/xception_FR_ffpp_test.parquet", index=False)

reals = scores[scores.label==0]
print("\n=== AUC per method (REENACTMENT-trained, VERIFIED) ===")
for m in ['faceswap','deepfakes','face2face','neuraltextures']:
    fk = scores[(scores.label==1)&(scores.method==m)]
    sub = pd.concat([reals, fk])
    print(f"  {m:16s}: AUC {roc_auc_score(sub.label, sub.prob_fake):.4f}")

  scored 64/22388
  scored 704/22388
  scored 1344/22388
  scored 1984/22388
  scored 2624/22388
  scored 3264/22388
  scored 3904/22388
  scored 4544/22388
  scored 5184/22388
  scored 5824/22388
  scored 6464/22388
  scored 7104/22388
  scored 7744/22388
  scored 8384/22388
  scored 9024/22388
  scored 9664/22388
  scored 10304/22388
  scored 10944/22388
  scored 11584/22388
  scored 12224/22388
  scored 12864/22388
  scored 13504/22388
  scored 14144/22388
  scored 14784/22388
  scored 15424/22388
  scored 16064/22388
  scored 16704/22388
  scored 17344/22388
  scored 17984/22388
  scored 18624/22388
  scored 19264/22388
  scored 19904/22388
  scored 20544/22388
  scored 21184/22388
  scored 21824/22388
scored 22388 frames

=== AUC per method (REENACTMENT-trained, VERIFIED) ===
  faceswap        : AUC 0.3766
  deepfakes       : AUC 0.6546
  face2face       : AUC 0.5064
  neuraltextures  : AUC 0.6536


In [61]:
import sys, importlib.util
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# switch back to calibration env (src-first, purge DFB metrics)
for k in list(sys.modules.keys()):
    if k=="metrics" or k.startswith("metrics.") or k in ("calibration","calibrate_scores"):
        del sys.modules[k]
sys.path = [p for p in sys.path if "DeepfakeBench" not in p]
if f"{REPO}/src" in sys.path: sys.path.remove(f"{REPO}/src")
sys.path.insert(0, f"{REPO}/src")

spec = importlib.util.spec_from_file_location("calibrate_scores", f"{REPO}/src/calibrate_scores.py")
cs = importlib.util.module_from_spec(spec); sys.modules["calibrate_scores"]=cs
spec.loader.exec_module(cs)

# calibrate FR scores — fit on neuraltextures (its best in-domain-ish reenactment method)
results_fr = cs.run_calibration_experiment(
    scores_path=f"{REPO}/reports/scores/xception_FR_ffpp_test.parquet",
    in_domain_method="neuraltextures",   # FR's strongest method
    B=2000,
)
import os
results_fr["pooled"].to_csv(f"{REPO}/reports/calibration/xception_FR_pooled.csv", index=False)
results_fr["coupling"].to_csv(f"{REPO}/reports/calibration/xception_FR_coupling.csv", index=False)

=== leakage-safe split (by identity) ===
  mode=group_disjoint  n_calib=11198  n_test=11190
  calib_pos_rate=0.800  test_pos_rate=0.800

=== R1/R2: pooled calibration (FF++ all methods, test split) ===
               set     n      ECE   ECE_lo   ECE_hi    Brier
raw (uncalibrated) 11190 0.693687 0.686504 0.699758 0.686999
 hybrid-calibrated 11190 0.206798 0.203059 0.224541 0.159764
  --> ECE reduced by 0.4869 (70.2%)

=== R3: forgery coupling (calibration fit on 'neuraltextures'+real, n_fit=4480) ===
  applied per-method to the TEST split:
    faceswap         n= 4476  AUC=0.387  ECE_raw=0.5141  ECE_cal=0.3261
    deepfakes        n= 4474  AUC=0.664  ECE_raw=0.4144  ECE_cal=0.2013
    face2face        n= 4479  AUC=0.495  ECE_raw=0.4547  ECE_cal=0.2624
    neuraltextures   n= 4478  AUC=0.640  ECE_raw=0.4134  ECE_cal=0.1582  <- in-domain

  INTERPRETATION: if ECE_cal stays low for the in-domain method but grows
  for reenactment methods, the calibration does NOT transfer across forgery
 

In [62]:
import os, glob, pandas as pd
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

print("=== reports/scores/ ===")
for f in sorted(glob.glob(f"{REPO}/reports/scores/*.parquet")):
    df = pd.read_parquet(f)
    print(f"  {os.path.basename(f):40s} {len(df):>6} rows, {os.path.getsize(f)/1e6:.1f}MB")

print("\n=== reports/calibration/ ===")
for f in sorted(glob.glob(f"{REPO}/reports/calibration/*.csv")):
    df = pd.read_csv(f)
    print(f"  {os.path.basename(f):40s} {len(df)} rows")
    print(f"      cols: {list(df.columns)}")

=== reports/scores/ ===
  xception_FR_ffpp_test.parquet             22388 rows, 0.8MB
  xception_ffpp_test.parquet                22388 rows, 0.8MB

=== reports/calibration/ ===
  xception_FR_coupling.csv                 4 rows
      cols: ['method', 'n', 'AUC', 'ECE_raw', 'ECE_cal']
  xception_FR_pooled.csv                   2 rows
      cols: ['set', 'n', 'ECE', 'ECE_lo', 'ECE_hi', 'Brier']
  xception_ffpp_coupling.csv               4 rows
      cols: ['method', 'n', 'AUC', 'ECE_raw', 'ECE_cal']
  xception_ffpp_pooled.csv                 2 rows
      cols: ['set', 'n', 'ECE', 'ECE_lo', 'ECE_hi', 'Brier']


In [63]:
import pandas as pd, numpy as np, json, os
from datetime import datetime
from scipy.stats import pearsonr, spearmanr
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# pool both checkpoints' coupling tables
fs = pd.read_csv(f"{REPO}/reports/calibration/xception_ffpp_coupling.csv"); fs["checkpoint"]="train_on_fs"; fs["fit_method"]="faceswap"
fr = pd.read_csv(f"{REPO}/reports/calibration/xception_FR_coupling.csv"); fr["checkpoint"]="train_on_fr"; fr["fit_method"]="neuraltextures"
both = pd.concat([fs, fr]).reset_index(drop=True)
print(both[["checkpoint","method","AUC","ECE_raw","ECE_cal"]].to_string(index=False))

r_p, p_p = pearsonr(both.AUC, both.ECE_cal)
r_s, p_s = spearmanr(both.AUC, both.ECE_cal)
print(f"\nAUC vs ECE_cal:  Pearson r={r_p:.3f} (p={p_p:.4f}) | Spearman ρ={r_s:.3f} (p={p_s:.4f})")

# save pooled table + a metadata sidecar
both.to_csv(f"{REPO}/reports/calibration/coupling_pooled_2ckpt.csv", index=False)
meta = {
    "experiment": "NB02_competence_calibration_coupling",
    "date": datetime.now().isoformat(),
    "datasets": {"eval": "FaceForensics++ c23 test split", "frames_per_ckpt": 22388},
    "checkpoints": ["train_on_fs/xception.pth", "train_on_fr/xception.pth"],
    "backbone": "xception (DeepfakeBench)",
    "calibrator": "HybridCalibrator (Platt n<1000 / isotonic n>=1000), Option D",
    "split": "leakage_safe group-disjoint by identity_id, calib_frac=0.5, seed=42",
    "metrics": "ECE equal-mass 15-bin, Brier, bootstrap CI B=2000 stratified",
    "finding": {
        "AUC_vs_ECEcal_pearson_r": round(float(r_p),4),
        "AUC_vs_ECEcal_pearson_p": round(float(p_p),4),
        "AUC_vs_ECEcal_spearman_rho": round(float(r_s),4),
        "interpretation": "calibration error rises as detection competence (AUC) falls; "
                          "calibration fit in a competent regime fails to transfer to incompetent regimes"
    },
    "per_ckpt_pooled": {
        "train_on_fs": {"raw_ECE": 0.4553, "cal_ECE": 0.0473, "reduction_pct": 89.6},
        "train_on_fr": {"raw_ECE": 0.6937, "cal_ECE": 0.2068, "reduction_pct": 70.2}
    }
}
with open(f"{REPO}/reports/calibration/NB02_findings.json","w") as f:
    json.dump(meta, f, indent=2)
print("\nsaved coupling_pooled_2ckpt.csv + NB02_findings.json (with provenance)")

 checkpoint         method      AUC  ECE_raw  ECE_cal
train_on_fs       faceswap 0.973808 0.043748 0.036445
train_on_fs      deepfakes 0.839122 0.237854 0.176211
train_on_fs      face2face 0.625260 0.411805 0.319473
train_on_fs neuraltextures 0.600081 0.431442 0.342808
train_on_fr       faceswap 0.386802 0.514096 0.326051
train_on_fr      deepfakes 0.664467 0.414418 0.201341
train_on_fr      face2face 0.494803 0.454652 0.262362
train_on_fr neuraltextures 0.640339 0.413413 0.158202

AUC vs ECE_cal:  Pearson r=-0.817 (p=0.0133) | Spearman ρ=-0.810 (p=0.0149)

saved coupling_pooled_2ckpt.csv + NB02_findings.json (with provenance)


In [64]:
import os
os.chdir(REPO)
!cp /content/drive/MyDrive/CDTS_Research/.git-credentials /root/.git-credentials 2>/dev/null
!git add reports/scores/xception_ffpp_test.parquet reports/scores/xception_FR_ffpp_test.parquet
!git add reports/calibration/
!git add src/inference.py src/calibrate_scores.py
!git status --short

Refresh index: 100% (37/37), done.
 m external/DeepfakeBench
 M notebooks/GBDF_download.ipynb
A  reports/calibration/NB02_findings.json
A  reports/calibration/coupling_pooled_2ckpt.csv
A  reports/calibration/xception_FR_coupling.csv
A  reports/calibration/xception_FR_pooled.csv
A  reports/calibration/xception_ffpp_coupling.csv
A  reports/calibration/xception_ffpp_pooled.csv
A  reports/scores/xception_FR_ffpp_test.parquet
A  reports/scores/xception_ffpp_test.parquet
A  src/calibrate_scores.py
A  src/inference.py


In [65]:
import os
os.chdir(REPO)
!git commit -m "NB02 first results: competence-calibration coupling (Pearson r=-0.82, p=0.013) across 2 Xception checkpoints x 4 FF++ forgery methods; ECE tracks AUC"
!git push origin main 2>&1 | tail -4
!git rev-parse HEAD

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@f179bda5aef8.(none)')
fatal: could not read Username for 'https://github.com': No such device or address
6d50d650162234da680ebe43b464768bd2a52638


In [66]:
import os, subprocess
os.chdir(REPO)
PARENT = "/content/drive/MyDrive/CDTS_Research"

# restore git identity + credentials from Drive (per your standard bootstrap)
subprocess.run(f'cp "{PARENT}/.gitconfig" /root/.gitconfig', shell=True)
subprocess.run(f'cp "{PARENT}/.git-credentials" /root/.git-credentials', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)

# verify identity is set
r = subprocess.run("git config --global user.name && git config --global user.email", shell=True, capture_output=True, text=True)
print("identity:", r.stdout.strip() if r.stdout.strip() else "STILL MISSING")

identity: Md Anas Biswas
anasbiswas@gmail.com


In [67]:
import os
os.chdir(REPO)
!git commit -m "NB02 first results: competence-calibration coupling (Pearson r=-0.82, p=0.013) across 2 Xception checkpoints x 4 FF++ forgery methods; ECE tracks AUC"
!git push origin main 2>&1 | tail -4
!git rev-parse HEAD

[main add3cd4] NB02 first results: competence-calibration coupling (Pearson r=-0.82, p=0.013) across 2 Xception checkpoints x 4 FF++ forgery methods; ECE tracks AUC
 10 files changed, 388 insertions(+)
 create mode 100644 reports/calibration/NB02_findings.json
 create mode 100644 reports/calibration/coupling_pooled_2ckpt.csv
 create mode 100644 reports/calibration/xception_FR_coupling.csv
 create mode 100644 reports/calibration/xception_FR_pooled.csv
 create mode 100644 reports/calibration/xception_ffpp_coupling.csv
 create mode 100644 reports/calibration/xception_ffpp_pooled.csv
 create mode 100644 reports/scores/xception_FR_ffpp_test.parquet
 create mode 100644 reports/scores/xception_ffpp_test.parquet
 create mode 100644 src/calibrate_scores.py
 create mode 100644 src/inference.py
To https://github.com/anasbiswas1/deepfake-trust-research.git
   6d50d65..add3cd4  main -> main
add3cd4be5a8404842ff7fe15e9560d47f63a956


In [68]:
import sys, importlib.util, os
import numpy as np, pandas as pd
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
method = "simswap"

# switch to calibration env (src first, purge DFB metrics)
for k in list(sys.modules.keys()):
    if k=="metrics" or k.startswith("metrics.") or k=="calibration":
        del sys.modules[k]
sys.path = [p for p in sys.path if "DeepfakeBench" not in p]
if f"{REPO}/src" in sys.path: sys.path.remove(f"{REPO}/src")
sys.path.insert(0, f"{REPO}/src")
import calibration as cal, metrics as met

scores = pd.read_parquet(f"{REPO}/reports/scores/xceptionFS_df40_{method}.parquet")
p = scores.prob_fake.values.astype(float); y = scores.label.values.astype(int)
groups = scores.identity_id.values if "identity_id" in scores.columns else None

ci, ti, info = cal.leakage_safe_split(y, groups=groups, calib_frac=0.5, seed=42)
p_cal, _ = cal.fit_predict("hybrid", p[ci], y[ci], p[ti], switch_threshold_n=1000)
ece_raw = met.ece(p[ti], y[ti], n_bins=15, scheme="equal_mass")
ece_cal = met.ece(p_cal, y[ti], n_bins=15, scheme="equal_mass")
auc = met.roc_auc(p[ti], y[ti]) if hasattr(met,"roc_auc") else float("nan")
print(f"{method}: AUC={auc:.4f}  ECE_raw={ece_raw:.4f}  ECE_cal={ece_cal:.4f}")

# append to DF40 coupling pool
row = {"checkpoint":"train_on_fs","method":f"df40_{method}","family":"FS",
       "n":int(len(ti)),"AUC":auc,"ECE_raw":ece_raw,"ECE_cal":ece_cal}
pool_path = f"{REPO}/reports/calibration/coupling_df40.csv"
pool = pd.read_csv(pool_path) if os.path.exists(pool_path) else pd.DataFrame()
pool = pd.concat([pool, pd.DataFrame([row])]).drop_duplicates("method", keep="last").reset_index(drop=True)
pool.to_csv(pool_path, index=False)
print(f"\nDF40 coupling pool: {len(pool)} methods")
print(pool[["method","AUC","ECE_cal"]].to_string(index=False))

simswap: AUC=0.9478  ECE_raw=0.0828  ECE_cal=0.0238

DF40 coupling pool: 1 methods
      method      AUC  ECE_cal
df40_simswap 0.947799 0.023779


In [ ]:
import os, glob, json, zipfile, shutil
import pandas as pd
import importlib.util, sys
from sklearn.metrics import roc_auc_score
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# data_prep loader
spec = importlib.util.spec_from_file_location("data_prep", f"{REPO}/src/data_prep.py")
data_prep = importlib.util.module_from_spec(spec); sys.modules["data_prep"]=data_prep
spec.loader.exec_module(data_prep)

# real index (Celeb-DF crop) — same for all methods, build once
CDF_REAL = f"{REPO}/data/frames/Celeb-DF-v2"
real_index = {"/".join(fp.split("/")[-2:]): fp
              for fp in glob.glob(f"{CDF_REAL}/**/frames/**/*.png", recursive=True)}
print(f"real index (Celeb-DF): {len(real_index)} frames\n")

# remaining core methods (simswap already done)
methods = ["inswap","blendface","fomm","facevid2vid","StyleGAN2","sd2.1","ddim","styleclip"]

for method in methods:
    print(f"=== {method} ===")
    zp = f"{REPO}/data/df40_core/test/{method}.zip"
    if not os.path.exists(zp):
        print(f"  zip missing, skip"); continue
    jpath = f"{REPO}/data/df40/dataset_json/{method}_cdf.json"
    if not os.path.exists(jpath):
        print(f"  {method}_cdf.json missing, skip"); continue

    # unzip to method-specific local dir
    fdir = f"/content/df40_{method}"
    if not os.path.isdir(fdir):
        os.makedirs(fdir, exist_ok=True)
        with zipfile.ZipFile(zp) as z: z.extractall(fdir)
    fake_index = {"/".join(fp.split("/")[-2:]): fp
                  for fp in glob.glob(f"{fdir}/**/*.png", recursive=True)}

    # manifest + remap
    df = data_prep.build_manifest_from_json(f"{method}_cdf", jpath, frames_root=None)
    def remap(row):
        key = "/".join(row["frame_path"].split("/")[-2:])
        return fake_index.get(key) if row["label"]==1 else real_index.get(key)
    df["frame_path"] = df.apply(remap, axis=1)
    df = df[df["frame_path"].notna()].reset_index(drop=True)
    bal = df['label'].value_counts().to_dict()
    print(f"  manifest: {len(df)} frames, labels {bal}")
    if df['label'].nunique() < 2:
        print(f"  SKIP: only one label"); shutil.rmtree(fdir, ignore_errors=True); continue

    # score (model + inference already loaded in env)
    scores = inference.score_manifest(model, device, df, batch_size=64, verbose=False)
    scores.to_parquet(f"{REPO}/reports/scores/xceptionFS_df40_{method}.parquet", index=False)
    auc = roc_auc_score(scores.label, scores.prob_fake)
    print(f"  AUC = {auc:.4f}  (n={len(scores)})")

    # cleanup local frames to save disk
    shutil.rmtree(fdir, ignore_errors=True)

print("\n=== Pass 1 done — all methods scored ===")
for f in sorted(glob.glob(f"{REPO}/reports/scores/xceptionFS_df40_*.parquet")):
    print("  ", os.path.basename(f))

real index (Celeb-DF): 16420 frames

=== inswap ===
